# Modelo Supervisado 2: XGBoost

Este modelo complementa el análisis abordando el objetivo de **clasificar el nivel de riesgo agrícola** (multiclase) de una siembra.
Se utiliza **XGBoost (Extreme Gradient Boosting)** por su robustez ante datos tabulares no lineales y su capacidad para manejar clases desbalanceadas mediante pesos (`sample_weight`).

## Fase 0: Configuración Inicial y Carga de Datos

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    f1_score, accuracy_score, log_loss,
    precision_score, recall_score,
    cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score,
    roc_auc_score, average_precision_score,
    roc_curve, auc, precision_recall_curve
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import label_binarize
from pathlib import Path

from src.data_loader import DataLoader
from src.xgboost_model import XGBoostDataPrep

# Configuraciones globales
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted')
Path('figuras').mkdir(exist_ok=True)

In [ ]:
# Carga de datos
loader = DataLoader('../data/raw/siap_2010_2024.csv')
try:
    df_raw = loader.cargar()
except FileNotFoundError:
    loader = DataLoader('../siap_procesado.csv')
    df_raw = loader.cargar()

print('Dimensiones del dataset:', df_raw.shape)
df_raw.dropna(subset=['Sembrada', 'Volumenproduccion'], inplace=True)

## Fase 1: Ingeniería de Features y Preparación de Datos

Se aplicará el preprocesamiento definido en `src/xgboost_model.py`:
1. Codificación de variables (LabelEncoding).
2. Creación de variables derivadas (`log_sembrada`, `interaccion_mod_ciclo`).
3. **Split Temporal**: 
   - Entrenamiento: 2010-2020
   - Validación: 2021-2022
   - Test: 2023-2024
4. Cálculo de pesos para el desbalance de clases.

In [ ]:
prep = XGBoostDataPrep(df_raw)
df_processed = prep.preprocess()

(X_train, y_train), (X_val, y_val), (X_test, y_test) = prep.temporal_split()

# Calcular sample weights para entrenamiento debido al desbalance extremo
sample_weights = prep.get_sample_weights(y_train)
target_names = list(prep.riesgo_map.keys())

## Fase 2: Modelo Baseline (XGBoost sin optimizar)

Entrenaremos un clasificador multiclase de XGBoost utilizando sus hiperparámetros por defecto para establecer una métrica de referencia (baseline).

In [ ]:
xgb_baseline = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    random_state=RANDOM_STATE,
    n_estimators=100,
    tree_method='hist'
)

print('Entrenando modelo Baseline...')
xgb_baseline.fit(
    X_train, y_train, 
    sample_weight=sample_weights,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False
)
print('¡Modelo Baseline entrenado!')

In [ ]:
y_val_pred = xgb_baseline.predict(X_val)

print('=== Reporte de Clasificación (Baseline en Validación) ===')
print(classification_report(y_val, y_val_pred, target_names=target_names))

f1_macro_base = f1_score(y_val, y_val_pred, average='macro')
acc_base = accuracy_score(y_val, y_val_pred)
print(f'F1-Score (Macro): {f1_macro_base:.4f}')
print(f'Accuracy: {acc_base:.4f}')

In [ ]:
cm_base = confusion_matrix(y_val, y_val_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_title('Matriz de Confusión - Modelo Baseline')
ax.set_ylabel('Valor Real')
ax.set_xlabel('Predicción')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('figuras/confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## Fase 3: Regularización con Gamma, Lambda (L2) y Alpha (L1)

XGBoost puede sobreajustarse a los datos de entrenamiento. Para contrarrestar esto, analizaremos el efecto de tres hiperparámetros de regularización:

- **Gamma ($\gamma$, `min_split_loss`)**: Reducción mínima de pérdida requerida para hacer una partición adicional en un nodo hoja. A mayor gamma, el algoritmo se vuelve más conservador, podando ramas que no aportan ganancia suficiente.

$$\text{Ganancia} = \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} - \gamma$$

- **Lambda ($\lambda$, `reg_lambda`, L2)**: Regularización L2 sobre los pesos de las hojas. Suaviza las predicciones reduciendo la magnitud cuadrada de los pesos, lo que disminuye la varianza del modelo.

- **Alpha ($\alpha$, `reg_alpha`, L1)**: Regularización L1 sobre los pesos de las hojas. Puede llevar algunos pesos exactamente a cero, produciendo modelos más simples (*sparsity*).

### Experimento 1: Efecto individual de Gamma

In [ ]:
gamma_values = [0, 0.1, 0.5, 1, 2, 5, 10]

gamma_results = []
for g in gamma_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=g, reg_lambda=1, reg_alpha=0
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    gamma_results.append({
        'gamma': g,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  gamma={g:>5} -> F1-macro={gamma_results[-1]["f1_macro"]:.4f}  logloss={gamma_results[-1]["logloss"]:.4f}')

df_gamma = pd.DataFrame(gamma_results)
print('\nResultados completos:')
display(df_gamma)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_gamma['gamma'], df_gamma['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Gamma en F1-Macro', fontsize=13)
axes[0].set_xlabel('Gamma')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_gamma['gamma'], df_gamma['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Gamma en Log Loss', fontsize=13)
axes[1].set_xlabel('Gamma')
axes[1].set_ylabel('Log Loss')

plt.tight_layout()
plt.savefig('figuras/exp1_gamma.png', dpi=150, bbox_inches='tight')
plt.show()

best_gamma = df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'gamma']
print(f'\nMejor gamma encontrado: {best_gamma}')

### Experimento 2: Efecto individual de Lambda (L2)

In [ ]:
lambda_values = [0, 0.01, 0.1, 1, 5, 10, 50, 100]

lambda_results = []
for l in lambda_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=best_gamma, reg_lambda=l, reg_alpha=0
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    lambda_results.append({
        'reg_lambda': l,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  lambda={l:>6} -> F1-macro={lambda_results[-1]["f1_macro"]:.4f}  logloss={lambda_results[-1]["logloss"]:.4f}')

df_lambda = pd.DataFrame(lambda_results)
print('\nResultados completos:')
display(df_lambda)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_lambda['reg_lambda'], df_lambda['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Lambda (L2) en F1-Macro', fontsize=13)
axes[0].set_xlabel('Lambda')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].set_xscale('symlog', linthresh=0.1)
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_lambda['reg_lambda'], df_lambda['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Lambda (L2) en Log Loss', fontsize=13)
axes[1].set_xlabel('Lambda')
axes[1].set_ylabel('Log Loss')
axes[1].set_xscale('symlog', linthresh=0.1)

plt.tight_layout()
plt.savefig('figuras/exp2_lambda.png', dpi=150, bbox_inches='tight')
plt.show()

best_lambda = df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'reg_lambda']
print(f'\nMejor lambda encontrado: {best_lambda}')

### Experimento 3: Efecto individual de Alpha (L1)

In [ ]:
alpha_values = [0, 0.001, 0.01, 0.1, 1, 5, 10, 50]

alpha_results = []
for a in alpha_values:
    model = xgb.XGBClassifier(
        objective='multi:softprob', num_class=5,
        random_state=RANDOM_STATE, n_estimators=100,
        tree_method='hist',
        gamma=best_gamma, reg_lambda=best_lambda, reg_alpha=a
    )
    model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)
    
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    
    alpha_results.append({
        'reg_alpha': a,
        'f1_macro': f1_score(y_val, y_pred, average='macro'),
        'accuracy': accuracy_score(y_val, y_pred),
        'logloss': log_loss(y_val, y_proba)
    })
    print(f'  alpha={a:>6} -> F1-macro={alpha_results[-1]["f1_macro"]:.4f}  logloss={alpha_results[-1]["logloss"]:.4f}')

df_alpha = pd.DataFrame(alpha_results)
print('\nResultados completos:')
display(df_alpha)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df_alpha['reg_alpha'], df_alpha['f1_macro'], marker='o', color='#3b7fc4', linewidth=2)
axes[0].set_title('Efecto de Alpha (L1) en F1-Macro', fontsize=13)
axes[0].set_xlabel('Alpha')
axes[0].set_ylabel('F1-Score (Macro)')
axes[0].set_xscale('symlog', linthresh=0.001)
axes[0].axhline(y=f1_macro_base, color='red', linestyle='--', label=f'Baseline ({f1_macro_base:.4f})')
axes[0].legend()

axes[1].plot(df_alpha['reg_alpha'], df_alpha['logloss'], marker='s', color='#e07b39', linewidth=2)
axes[1].set_title('Efecto de Alpha (L1) en Log Loss', fontsize=13)
axes[1].set_xlabel('Alpha')
axes[1].set_ylabel('Log Loss')
axes[1].set_xscale('symlog', linthresh=0.001)

plt.tight_layout()
plt.savefig('figuras/exp3_alpha.png', dpi=150, bbox_inches='tight')
plt.show()

best_alpha = df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'reg_alpha']
print(f'\nMejor alpha encontrado: {best_alpha}')

### Experimento 4: Búsqueda conjunta (RandomizedSearchCV)

Realizamos una búsqueda aleatoria combinando gamma, lambda, alpha junto con otros hiperparámetros clave del modelo.

In [ ]:
param_dist = {
    'gamma': [0, 0.1, 0.5, 1, 2, 5],
    'reg_lambda': [0.01, 0.1, 1, 5, 10, 50],
    'reg_alpha': [0, 0.01, 0.1, 1, 5, 10],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

xgb_search = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    random_state=RANDOM_STATE,
    tree_method='hist'
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=xgb_search,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    return_train_score=True
)

print('Iniciando busqueda de hiperparametros (esto puede tardar varios minutos)...')
random_search.fit(X_train, y_train, sample_weight=sample_weights)

print(f'\nMejor F1-Macro CV: {random_search.best_score_:.4f}')
print(f'Mejores hiperparametros:')
for param, val in random_search.best_params_.items():
    print(f'  {param}: {val}')

In [ ]:
cv_results = pd.DataFrame(random_search.cv_results_)
cols_show = ['rank_test_score', 'mean_test_score', 'std_test_score',
             'param_gamma', 'param_reg_lambda', 'param_reg_alpha',
             'param_max_depth', 'param_learning_rate', 'param_n_estimators']
top10 = cv_results[cols_show].sort_values('rank_test_score').head(10)
print('=== Top 10 configuraciones ===')
display(top10)

### Tabla comparativa de los 4 experimentos

In [ ]:
resumen = pd.DataFrame({
    'Configuracion': ['Baseline (defaults)', 
                      f'Mejor Gamma (g={best_gamma})',
                      f'Mejor Lambda (l={best_lambda})',
                      f'Mejor Alpha (a={best_alpha})',
                      'RandomizedSearchCV (conjunto)'],
    'F1-Macro': [
        f1_macro_base,
        df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'f1_macro'],
        df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'f1_macro'],
        df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'f1_macro'],
        random_search.best_score_
    ],
    'Log Loss': [
        log_loss(y_val, xgb_baseline.predict_proba(X_val)),
        df_gamma.loc[df_gamma['f1_macro'].idxmax(), 'logloss'],
        df_lambda.loc[df_lambda['f1_macro'].idxmax(), 'logloss'],
        df_alpha.loc[df_alpha['f1_macro'].idxmax(), 'logloss'],
        np.nan
    ]
})

print('=== Resumen Comparativo de Regularizacion ===')
display(resumen)

## Fase 4: Modelo Final Optimizado y Evaluación Completa

Entrenamos el modelo con los **mejores hiperparámetros** encontrados en la búsqueda conjunta.
Combinamos los conjuntos de entrenamiento y validación para maximizar la cantidad de datos de entrenamiento,
y evaluamos exclusivamente sobre el conjunto de **test** (2023-2024), datos nunca vistos durante el entrenamiento ni la selección de hiperparámetros.

In [ ]:
# Obtener los mejores hiperparametros del RandomizedSearchCV
best_params = random_search.best_params_.copy()
best_params['objective'] = 'multi:softprob'
best_params['num_class'] = 5
best_params['random_state'] = RANDOM_STATE
best_params['tree_method'] = 'hist'

print('Hiperparametros del modelo final:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

# Combinar Train + Val para entrenamiento final
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])
sw_full = prep.get_sample_weights(y_train_full)

# Entrenar modelo final
xgb_final = xgb.XGBClassifier(**best_params)
xgb_final.fit(X_train_full, y_train_full, sample_weight=sw_full, verbose=False)
print('\nModelo final entrenado!')

In [ ]:
# Predicciones en Test
y_test_pred = xgb_final.predict(X_test)
y_test_proba = xgb_final.predict_proba(X_test)

print('=== Reporte de Clasificacion (Test 2023-2024) ===')
print(classification_report(y_test, y_test_pred, target_names=target_names))

# Metricas completas
metrics_final = {
    'Accuracy': accuracy_score(y_test, y_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_test_pred),
    'F1-Macro': f1_score(y_test, y_test_pred, average='macro'),
    'F1-Weighted': f1_score(y_test, y_test_pred, average='weighted'),
    'Precision-Macro': precision_score(y_test, y_test_pred, average='macro'),
    'Recall-Macro': recall_score(y_test, y_test_pred, average='macro'),
    'Log Loss': log_loss(y_test, y_test_proba),
    'Cohen Kappa': cohen_kappa_score(y_test, y_test_pred),
    'MCC': matthews_corrcoef(y_test, y_test_pred),
    'AUC-ROC (OvR macro)': roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro'),
}

print('\n=== Metricas del Modelo Final ===')
for name, val in metrics_final.items():
    print(f'  {name:25s}: {val:.4f}')

In [ ]:
# Matriz de confusion (absoluta y normalizada)
cm_final = confusion_matrix(y_test, y_test_pred)
cm_norm = cm_final.astype('float') / cm_final.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title('Matriz de Confusion (Absoluta)', fontsize=13)
axes[0].set_ylabel('Valor Real')
axes[0].set_xlabel('Prediccion')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title('Matriz de Confusion (Normalizada)', fontsize=13)
axes[1].set_ylabel('Valor Real')
axes[1].set_xlabel('Prediccion')

for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('figuras/confusion_matrix_final.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Curvas ROC y Precision-Recall por clase
classes = sorted(y_test.unique())
y_test_bin = label_binarize(y_test, classes=classes)

colores = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_test_proba[:, i])
    roc_auc_val = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=colores[i], linewidth=2,
                 label=f'{target_names[i]} (AUC={roc_auc_val:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('Curvas ROC por Clase (One-vs-Rest)', fontsize=13)
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].legend(loc='lower right', fontsize=8)

for i, cls in enumerate(classes):
    prec, rec, _ = precision_recall_curve(y_test_bin[:, i], y_test_proba[:, i])
    pr_auc_val = auc(rec, prec)
    axes[1].plot(rec, prec, color=colores[i], linewidth=2,
                 label=f'{target_names[i]} (AUC-PR={pr_auc_val:.3f})')

axes[1].set_title('Curvas Precision-Recall por Clase', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('figuras/roc_pr_curves_final.png', dpi=150, bbox_inches='tight')
plt.show()

### Comparación Baseline vs. Modelo Optimizado

In [ ]:
# Evaluar baseline tambien en test para comparar
y_test_pred_base = xgb_baseline.predict(X_test)
y_test_proba_base = xgb_baseline.predict_proba(X_test)

comparacion = pd.DataFrame({
    'Metrica': list(metrics_final.keys()),
    'Baseline': [
        accuracy_score(y_test, y_test_pred_base),
        balanced_accuracy_score(y_test, y_test_pred_base),
        f1_score(y_test, y_test_pred_base, average='macro'),
        f1_score(y_test, y_test_pred_base, average='weighted'),
        precision_score(y_test, y_test_pred_base, average='macro'),
        recall_score(y_test, y_test_pred_base, average='macro'),
        log_loss(y_test, y_test_proba_base),
        cohen_kappa_score(y_test, y_test_pred_base),
        matthews_corrcoef(y_test, y_test_pred_base),
        roc_auc_score(y_test, y_test_proba_base, multi_class='ovr', average='macro'),
    ],
    'Optimizado': list(metrics_final.values())
})

comparacion['Delta'] = comparacion['Optimizado'] - comparacion['Baseline']
comparacion['% Mejora'] = ((comparacion['Optimizado'] - comparacion['Baseline']) / comparacion['Baseline'].abs() * 100).round(2)

print('=== Comparacion Baseline vs. Modelo Optimizado (Test) ===')
display(comparacion)

In [ ]:
# Grafico de barras comparativo
metricas_plot = ['Accuracy', 'Balanced Accuracy', 'F1-Macro', 'Precision-Macro', 'Recall-Macro', 'Cohen Kappa']
comp_plot = comparacion[comparacion['Metrica'].isin(metricas_plot)].copy()

x = np.arange(len(metricas_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, comp_plot['Baseline'], width, label='Baseline', color='#c0c0c0', edgecolor='white')
bars2 = ax.bar(x + width/2, comp_plot['Optimizado'], width, label='Optimizado', color='#3b7fc4', edgecolor='white')

ax.set_ylabel('Score')
ax.set_title('Comparacion: Baseline vs. Modelo Optimizado (Test)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metricas_plot, rotation=30, ha='right')
ax.legend()
ax.set_ylim(0, 1.05)
ax.spines[['top', 'right']].set_visible(False)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('figuras/comparacion_baseline_optimizado.png', dpi=150, bbox_inches='tight')
plt.show()

## Fase 5: Interpretabilidad del Modelo

Entenderemos que variables fueron las mas determinantes para predecir el riesgo agricola,
utilizando tanto la importancia nativa de XGBoost como los valores SHAP (SHapley Additive exPlanations).

### Feature Importance (nativa de XGBoost)

In [ ]:
# Feature Importance con 3 metricas diferentes
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, imp_type in enumerate(['weight', 'gain', 'cover']):
    xgb.plot_importance(xgb_final, importance_type=imp_type, ax=axes[i],
                       title=f'Importancia ({imp_type})', xlabel=imp_type.capitalize())

plt.tight_layout()
plt.savefig('figuras/feature_importance_3metricas.png', dpi=150, bbox_inches='tight')
plt.show()

### SHAP Values (SHapley Additive exPlanations)

SHAP nos permite entender no solo *que* variables son importantes,
sino *como* y en *que direccion* influyen en cada prediccion individual.

In [ ]:
import shap

# Crear el explainer para el modelo XGBoost
explainer = shap.TreeExplainer(xgb_final)

# Calcular SHAP values sobre una muestra del test (para eficiencia)
sample_size = min(5000, len(X_test))
X_test_sample = X_test.sample(n=sample_size, random_state=RANDOM_STATE)
shap_values = explainer.shap_values(X_test_sample)

print(f'SHAP values calculados para {sample_size} muestras.')
print(f'Forma de shap_values: {len(shap_values)} clases, cada una con shape {shap_values[0].shape}')

In [ ]:
# Summary Plot (Bar): importancia media absoluta por clase
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_sample, plot_type='bar',
                  class_names=target_names, show=False)
plt.title('SHAP: Importancia Media Absoluta por Clase', fontsize=13)
plt.tight_layout()
plt.savefig('figuras/shap_bar_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary Plot (Beeswarm) para la clase mas interesante: Perdida Critica (clase 4)
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values[4], X_test_sample, show=False)
plt.title('SHAP Beeswarm: Perdida Critica (clase 4)', fontsize=13)
plt.tight_layout()
plt.savefig('figuras/shap_beeswarm_perdida_critica.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dependence plots para las 3 features mas importantes
mean_abs_shap = np.abs(shap_values[4]).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[-3:][::-1]
top3_features = [X_test_sample.columns[i] for i in top3_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(top3_features):
    shap.dependence_plot(feat, shap_values[4], X_test_sample, ax=axes[i], show=False)
    axes[i].set_title(f'Dependencia: {feat}', fontsize=11)

plt.suptitle('SHAP Dependence Plots - Perdida Critica', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('figuras/shap_dependence_top3.png', dpi=150, bbox_inches='tight')
plt.show()

### Analisis de Errores

Analizamos los patrones de predicciones incorrectas para entender las limitaciones del modelo.

In [ ]:
# Analisis de errores
errores_mask = y_test_pred != y_test.values
n_errores = errores_mask.sum()
pct_errores = n_errores / len(y_test) * 100

print(f'Total de errores: {n_errores} de {len(y_test)} ({pct_errores:.2f}%)')
print()

# Tipo de confusiones mas frecuentes
df_errores = pd.DataFrame({
    'Real': y_test.values[errores_mask],
    'Predicho': y_test_pred[errores_mask]
})
df_errores['Real_nombre'] = df_errores['Real'].map(prep.riesgo_map_inv)
df_errores['Predicho_nombre'] = df_errores['Predicho'].map(prep.riesgo_map_inv)

confusiones = df_errores.groupby(['Real_nombre', 'Predicho_nombre']).size().sort_values(ascending=False).head(10)
print('=== Top 10 Confusiones mas Frecuentes ===')
display(confusiones.reset_index(name='Frecuencia'))

In [ ]:
# F1-Score por clase: grafico de barras para visualizar clases debiles
f1_por_clase = f1_score(y_test, y_test_pred, average=None)

fig, ax = plt.subplots(figsize=(10, 5))
colores_riesgo = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']
bars = ax.bar(target_names, f1_por_clase, color=colores_riesgo, edgecolor='white', width=0.6)

for bar, val in zip(bars, f1_por_clase):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('F1-Score por Nivel de Riesgo (Test)', fontsize=13)
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.1)
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('figuras/f1_por_clase.png', dpi=150, bbox_inches='tight')
plt.show()

### Conclusiones de Interpretabilidad

Los resultados de SHAP y feature importance nos permiten extraer las siguientes conclusiones:

1. **Variables mas predictivas**: Las variables con mayor importancia SHAP revelan cuales son los factores que mas influyen en la prediccion del riesgo agricola.
2. **Modalidad hidrica**: Se puede observar si la modalidad (riego vs temporal) es un factor determinante, lo cual es consistente con la hipotesis del proyecto.
3. **Cultivos de alto riesgo**: El encoding de cultivos permite identificar si ciertos tipos de cultivo son inherentemente mas vulnerables.
4. **Tendencia temporal**: La variable Anio nos indica si existe una tendencia creciente o decreciente en el riesgo a lo largo del tiempo.

## Fase 6: Validacion Cruzada Final y Robustez

Verificamos la estabilidad del modelo mediante validacion cruzada estratificada
y analizamos si el modelo se beneficiaria de mas datos o si ya convergio.

### Cross-Validation Estratificado

In [ ]:
from sklearn.model_selection import cross_validate

# Usar los mejores hiperparametros encontrados
xgb_cv = xgb.XGBClassifier(**best_params)

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring_cv = {
    'f1_macro': 'f1_macro',
    'accuracy': 'accuracy',
    'roc_auc_ovr': 'roc_auc_ovr',
    'neg_log_loss': 'neg_log_loss'
}

# Combinar todos los datos (train + val + test) para CV completo
X_all = pd.concat([X_train, X_val, X_test])
y_all = pd.concat([y_train, y_val, y_test])
sw_all = prep.get_sample_weights(y_all)

print('Ejecutando 5-Fold Stratified Cross-Validation...')
cv_results_final = cross_validate(
    xgb_cv, X_all, y_all,
    cv=cv_strat,
    scoring=scoring_cv,
    fit_params={'sample_weight': sw_all},
    return_train_score=True,
    n_jobs=-1
)

print('\n=== Resultados Cross-Validation (5-Fold) ===')
for metric in ['f1_macro', 'accuracy', 'roc_auc_ovr', 'neg_log_loss']:
    test_key = f'test_{metric}'
    train_key = f'train_{metric}'
    test_vals = cv_results_final[test_key]
    train_vals = cv_results_final[train_key]
    print(f'  {metric}:')
    print(f'    Test:  {test_vals.mean():.4f} +/- {test_vals.std():.4f}')
    print(f'    Train: {train_vals.mean():.4f} +/- {train_vals.std():.4f}')

In [ ]:
# Visualizar estabilidad entre folds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1-Macro por fold
folds = range(1, 6)
axes[0].bar(folds, cv_results_final['test_f1_macro'], color='#3b7fc4', edgecolor='white', label='Test')
axes[0].axhline(y=cv_results_final['test_f1_macro'].mean(), color='red', linestyle='--',
                label=f'Media: {cv_results_final["test_f1_macro"].mean():.4f}')
axes[0].set_title('F1-Macro por Fold', fontsize=13)
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('F1-Macro')
axes[0].legend()

# Comparacion Train vs Test por fold
x = np.arange(len(folds))
w = 0.35
axes[1].bar(x - w/2, cv_results_final['train_f1_macro'], w, label='Train', color='#c0c0c0', edgecolor='white')
axes[1].bar(x + w/2, cv_results_final['test_f1_macro'], w, label='Test', color='#3b7fc4', edgecolor='white')
axes[1].set_title('Train vs Test F1-Macro por Fold', fontsize=13)
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('F1-Macro')
axes[1].set_xticks(x)
axes[1].set_xticklabels(folds)
axes[1].legend()

plt.tight_layout()
plt.savefig('figuras/cross_validation_folds.png', dpi=150, bbox_inches='tight')
plt.show()

### Curvas de Aprendizaje

Analizamos si el modelo se beneficiaria de mas datos o si ya alcanzo su capacidad maxima.

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes_abs, train_scores, test_scores = learning_curve(
    xgb.XGBClassifier(**best_params),
    X_all, y_all,
    train_sizes=[0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    scoring='f1_macro',
    n_jobs=-1,
    fit_params={'sample_weight': sw_all[:len(X_all)]}
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.1, color='#3b7fc4')
ax.fill_between(train_sizes_abs, test_mean - test_std, test_mean + test_std, alpha=0.1, color='#e07b39')
ax.plot(train_sizes_abs, train_mean, 'o-', color='#3b7fc4', linewidth=2, label='Train')
ax.plot(train_sizes_abs, test_mean, 's-', color='#e07b39', linewidth=2, label='Validacion')

ax.set_title('Curva de Aprendizaje (F1-Macro)', fontsize=13)
ax.set_xlabel('Numero de muestras de entrenamiento')
ax.set_ylabel('F1-Score (Macro)')
ax.legend(loc='lower right')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('figuras/learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# Diagnostico
gap = train_mean[-1] - test_mean[-1]
print(f'Gap Train-Test final: {gap:.4f}')
if gap > 0.05:
    print('-> El modelo muestra signos de overfitting. Podria beneficiarse de mas regularizacion.')
elif test_mean[-1] - test_mean[-2] > 0.005:
    print('-> El modelo aun mejora con mas datos. Podria beneficiarse de un dataset mas grande.')
else:
    print('-> El modelo ha convergido. Mas datos no mejorarian significativamente el rendimiento.')

### Discusion y Limitaciones

**Fortalezas del modelo:**
- XGBoost con regularizacion (gamma, lambda, alpha) logra un balance entre complejidad y generalizacion.
- El uso de `sample_weight` permite manejar el desbalance extremo (95% clase mayoritaria) sin alterar los datos originales.
- El split temporal simula condiciones reales de uso: entrenar con datos historicos para predecir el futuro.

**Limitaciones:**
- El modelo solo utiliza variables del dataset SIAP. No incorpora datos climaticos externos (precipitacion, temperatura, indice de sequia) que podrian mejorar significativamente la prediccion.
- La codificacion de cultivos mediante LabelEncoding no captura relaciones semanticas entre cultivos similares.
- El desbalance extremo de clases sigue siendo un reto: las clases minoritarias (Riesgo alto, Perdida critica) son las mas dificiles de predecir correctamente.

**Posibles mejoras futuras:**
- Integrar datos meteorologicos del SMN (Servicio Meteorologico Nacional).
- Usar Target Encoding o CatBoost Encoding en lugar de LabelEncoding para variables de alta cardinalidad.
- Explorar tecnicas de ensemble: combinar XGBoost con otros modelos (LightGBM, Random Forest).
- Implementar un enfoque de regresion complementario para predecir la proporcion de siniestro de forma continua.